# Full Photometry Pipeline

This tutorial demonstrates how to run the end--to--end photometry pipeline on
real JWST data included with *Mophongo*. We build templates from the F444W
mosaic and fit fluxes in the F770W image using spatially varying PSF
matching kernels.

## Detect sources in F444W

In [ ]:
from pathlib import Path
import numpy as np
from astropy.io import fits
import logging
#logging.basicConfig(level=logging.ERROR)

miri_filt = '770'
version = 'v0.1'

img_dir = Path('/Users/ivo/Astro/PROJECTS/MINERVA/data/v2/')
field='uds-grizli-v8.0-minerva-v2.2-40mas'
field_miri = 'uds-sbkgsub-v2.3-80mas'
sci_444 = img_dir / f'{field}-f444w-clear_drc_sci.fits'
sci_miri = img_dir / f'{field_miri}-f{miri_filt}w_drz_sci.fits'


## Build PSF region map

In [ ]:
from mophongo.psf import DrizzlePSF
from mophongo.psf_map import PSFRegionMap
import glob

csv444 = glob.glob(str(img_dir)+'/uds*444*_wcs.csv')[0]
csvmiri = glob.glob(str(img_dir)+f'/uds*{miri_filt}*_wcs.csv')[0]

# initialize drizzler; also reads the associated _wcs.csv files (rate files wcs/header information)
dpsf_444 = DrizzlePSF(driz_image=str(sci_444), csv_file=csv444)
dpsf_miri = DrizzlePSF(driz_image=str(sci_miri), csv_file=csvmiri)

# map unique detector overlaps; keep only footprints overlapping the target mosaic
prm_444 = PSFRegionMap.from_footprints(dpsf_444.footprint, name='F444W').overlay_with(dpsf_444.driz_footprint)
prm_miri = PSFRegionMap.from_footprints(dpsf_miri.footprint, name=f'F{miri_filt}W').overlay_with(dpsf_miri.driz_footprint)

# compute overlay regions unique to both PSFs
prm_kern = prm_444.overlay_with(prm_miri)
prm_444.plot()
prm_miri.plot()
prm_kern.plot()
prm_kern.regions

## Create PSF kernels

In [ ]:
import mophongo.utils as utils

psf_dir = Path('../data/PSF')
stpsf_444 = 'UDS_NRC.._F444W_OS4_GRID25'
stpsf_miri = f'UDS_MIRI_F{miri_filt}W_OS4_GRID9'
size = 2.0

if not (img_dir / f'{field_miri}-f{miri_filt}w_psf.geojson').exists():
    # centroid positions of the regions: DONT drop poins here bc PSF list will be off
    pos = [np.squeeze(p.xy) for p in prm_kern.regions.geometry.centroid]

    # load webb psfs
    dpsf_444.epsf_obj.load_jwst_stdpsf(local_dir=str(psf_dir),
                                       filter_pattern=stpsf_444)
    dpsf_miri.epsf_obj.load_jwst_stdpsf(local_dir=str(psf_dir),
                                        filter_pattern=stpsf_miri)

    # drizzle at centroid positions, size of stamp in arcsec
    prm_444.psfs = dpsf_444.get_psf_radec(pos, size=size)
    prm_miri.psfs = dpsf_miri.get_psf_radec(pos, size=size)

    # store the PSFs + region maps
    prm_444.to_file(img_dir / f'{field}-f444w_psf.geojson')
    prm_miri.to_file(img_dir / f'{field_miri}-f{miri_filt}w_psf.geojson')

    # match kernels
    # @@@ need a better way to determine best fft window shape
    # @@@ No PSF found, position possibly outside footprint for 34.38666666666666, -5.243333333333333 in filter UDS_MIRI_F770W_OS4_GRID9. Returning empty output.
    # in this case, return nearest PSF, not empty 
    pixel_ratio = round(dpsf_miri.driz_pscale/dpsf_444.driz_pscale)

    # @@@ optional add gaussian_filter(X, 0.1)
    kernels = [ utils.matching_kernel(psf_444, psf_miri, recenter=True, pixel_ratio=pixel_ratio) 
                for psf_444, psf_miri in zip(prm_444.psfs, prm_miri.psfs) ]

    prm_kern.psfs = np.asarray(kernels)
    prm_kern.to_file(img_dir / f'{field_miri}-f444w_kernel_f{miri_filt}w.geojson')


## Run photometry

In [1]:
from pathlib import Path
from astropy.wcs import WCS
import astropy.units as u
from astropy.coordinates import SkyCoord
from shapely.geometry import Polygon
from shapely import points
from astropy.table import Table
from astropy.io import fits
from astropy.table import Table
from mophongo.psf_map import PSFRegionMap
from mophongo.fit import FitConfig
from mophongo.catalog import Catalog, get_bg_and_ivar
from mophongo.pipeline import Pipeline

# for testing, first run on small patch r < 0.5 arcmin
#r_trial = 0.5

miri_filt = '770'
out_root = Path('uds_'+miri_filt)
out_root.mkdir(exist_ok=True)

img_dir = Path('/Users/ivo/Astro/PROJECTS/MINERVA/data/v2/')
cat_dir = Path('/Users/ivo/Astro/PROJECTS/MINERVA/data/n2.2_m2.0_v1.0/')

cat_file = cat_dir / 'MINERVA-UDS_n2.2_m2.0_v1.0_LW_Kf444w_SUPER_CATALOG.fits'
fseg_LW = cat_dir / 'LW_f277w-f356w-f444w_SEGMAP.fits'
cat = Table.read(cat_file)

field='uds-grizli-v8.0-minerva-v2.2-40mas'
field_miri = 'uds-sbkgsub-v2.3-80mas'
miri_ext = '_drz'
nircam_ext = '-clear_drc'

# PSF + kernel maps
prm_444 = PSFRegionMap.from_geojson(str(img_dir / f'{field}-f444w_psf.geojson'))
prm_miri = PSFRegionMap.from_geojson(str(img_dir / f'{field_miri}-f{miri_filt}w_psf.geojson'))
prm_kern = PSFRegionMap.from_geojson(str(img_dir / f'{field_miri}-f444w_kernel_f{miri_filt}w.geojson'))

# NIRCam images
#fsci_444 = img_dir / f'{field}-f444w{nircam_ext}_sci.fits'
fsci_444 = cat_dir / 'LW_f277w-f356w-f444w_KRON_Kf444w_optavg.fits'
wcs_444 = WCS(fits.getheader(fsci_444))
# MIRI images
fsci_miri = img_dir / f'{field_miri}-f{miri_filt}w{miri_ext}_sci.fits'
fwht_miri = img_dir / f'{field_miri}-f{miri_filt}w{miri_ext}_wht.fits'
wcs_miri = WCS(fits.getheader(fsci_miri))

# load images + segmap
tmpl_444 = fits.getdata(fsci_444)
sci_miri = fits.getdata(fsci_miri)
wht_miri = fits.getdata(fwht_miri)
segmap = fits.getdata(fseg_LW)

# background and inverse variance calibration
bg_miri, ivar_miri, = get_bg_and_ivar(sci_miri, wht_miri, bg_filter_sigma=64.0)


/Users/ivo/Astro/PROJECTS/MOPHONGO/mophongo/mophongo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO: NumExpr defaulting to 10 threads.
Set DATE-AVG to '2023-02-02T21:15:02.856' from MJD-AVG.
Set DATE-END to '2024-01-19T03:07:41.308' from MJD-END'. [astropy.wcs.wcs]
Set DATE-AVG to '2023-02-02T21:15:02.856' from MJD-AVG.
Set DATE-END to '2024-01-19T03:07:41.308' from MJD-END'.
Set OBSGEO-B to     7.991511 from OBSGEO-[XYZ].
Set OBSGEO-H to 1680943524.537 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set OBSGEO-B to     7.991511 from OBSGEO-[XYZ].
Set OBSGEO-H to 1680943524.537 from OBSGEO-[XYZ]'.


In [2]:
if locals().get('r_trial', 0) > 0:
    #    run only on small subsection of the catalog for testing
    coords = SkyCoord(ra=cat['ra'], dec=cat['dec'])
    ref = SkyCoord(ra=34.4*u.deg, dec=-5.14*u.deg)
    mask = coords.separation(ref) < r_trial * u.arcmin
    cat = cat[mask]

# first fit, no shifts: first image is template, 2nd and on the fitting images
config = FitConfig(fit_astrometry_niter=2,
                   scene_minimum_bright=10,
                   aperture_diam=0.5,
                   aperture_catalog='use_aper',
                   generate_scene_catalog=True)

pipe = Pipeline([tmpl_444, sci_miri-bg_miri],
                segmap,
                weights=[None, ivar_miri],
                catalog=cat,
                psfs=[None, prm_miri],
                kernels=[None, prm_kern],
                wcs=[wcs_444, wcs_miri])

table, res = pipe.run(config=config)


Pipeline (init) memory: 5.6 GB
Pipeline (start) memory: 5.6 GB
Pipeline config: FitConfig(positivity=True, reg=0.0, bad_value=nan, solve_method='scene', cg_kwargs={'M': None, 'maxiter': 500, 'atol': 1e-06}, fit_covariances=False, fft_fast=False, fit_astrometry_niter=2, fit_astrometry_joint=True, reg_astrom=0.0001, snr_thresh_astrom=15.0, astrom_model='gp', astrom_centroid='centroid', astrom_kwargs={'poly': {'order': 0}, 'gp': {'length_scale': 400}}, multi_tmpl_chi2_thresh=5.0, multi_tmpl_psf_core=False, multi_tmpl_colour=False, multi_resolution_method='upsample', normal='tree', scene_merge_small=True, scene_minimum_bright=10, negative_snr_thresh=-1.0, aperture_diam=0.5, aperture_catalog='use_aper', aperture_units='arcsec', block_size=64, run_scene_solver=True, scene_coupling_thresh=0.001, generate_scene_catalog=True)


Extracting templates: 100%|██████████| 188775/188775 [11:31<00:00, 272.95it/s]


Pipepline: 188749 extracted templates, dropped 26.
Pipeline (templates) memory: 7.9 GB
Using kernel lookup table uds-sbkgsub-v2.3-80mas-f444w_kernel_f770w
upsampling image 1 by factor 2
Pruned 99989 templates with low L2 norm on weight map.


Convolving templates: 100%|██████████| 88760/88760 [00:26<00:00, 3362.20it/s]


Pipeline (convolved) memory: 10.3 GB
Wrote scene catalog scene_catalog_1.ecsv


SystemExit: 

/Users/ivo/Astro/PROJECTS/MOPHONGO/mophongo/mophongo/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [3]:
scene_cat = Table.read('scene_catalog_1.ecsv')
scene_cat['minerva_link'] = [
    f"https://minerva.colorado.edu/?ra={ra}&dec={dec}&zoom=8"
    for ra, dec in zip(scene_cat['ra'], scene_cat['dec'])
]
scene_cat.write(out_root / (str(out_root) + '_scene_catalog.csv'), overwrite=True)

In [ ]:
from matplotlib import pyplot as plt
scenes = pipe.all_scenes[0]
for i in range(len(scenes)):
    print(f'scene id {scenes[i].id}, sources {len(scenes[i].templates)}, bright {scenes[i].is_bright.sum()}')
    fig, ax = scenes[i].plot(tmpl_444, segmap, display_sig=5)
    fig.savefig(out_root / (str(out_root) + f'_scene_{scenes[i].id}.png') , dpi=300)
    plt.close(fig)

In [ ]:
#x,y = scenes[0].shift_basis[1]
#wcs_444.wcs_pix2world([scenes[0].shift_basis[1]], 0)    

In [ ]:
fits.writeto(out_root / (str(out_root) + '_residual.fits'), res[0], fits.getheader(fsci_miri), overwrite=True)
table.write(out_root / (str(out_root) + '_fit_table.fits'), overwrite=True)


In [ ]:
table

In [ ]:
# from astropy.coordinates import SkyCoord
# import astropy.units as u
# import matplotlib.pyplot as plt
# id	x	y	ra	dec
#ivar_770, bg_770 = get_bg_and_ivar(sci_770, wht_770, bg_filter_sigma=64.0)
# s=0.1
# #im = ivar_770[8000:10_000, 4000:6000]
# sl = slice(8000, 10_000), slice(4000, 6000)
# im = bg_770[sl]
# bg_770, ivar_770 = get_bg_and_ivar(sci_770[sl], wht_770[sl], bg_filter_sigma=64.0)

# #plt.imshow(bg_770, origin='lower', vmin=-s, vmax=s, cmap='gray')
# plt.imshow(ivar_770, origin='lower', vmin=0, vmax=10, cmap='gray')
#cat770 = Catalog.from_fits(sci_770, wht_770, estimate_background=True, estimate_ivar=True)
#cat770.plot_bg(nbin=1, fac=1)

# TESTING

In [ ]:
from pathlib import Path
from mophongo.psf_map import PSFRegionMap
from mophongo.psf import DrizzlePSF
from mophongo.psf_map import PSFRegionMap
import mophongo.utils as utils
import numpy as np
import glob

data_dir = Path('../data')
field = 'uds-test'
#field = 'uds-test2'
#field = 'uds-medium'
#field = 'uds-large'
#field = 'uds-half'
#field = 'uds-ahstar'
#field = 'uds-ahtwin'
#field = 'uds-bright'

field_miri = field+'-80mas'
#field_miri = field+'-40mas'
#field_miri = field 

fsci_444 = data_dir / f'{field}-f444w_sci.fits'
#wht_444 = data_dir / f'{field}-f444w_wht.fits'
fseg_LW = data_dir / f'{field}-LW_seg.fits'

# MIRI images
field_miri = field+'-80mas'
fsci_770 = data_dir / f'{field_miri}-f770w_sci.fits'
fwht_770 = data_dir / f'{field_miri}-f770w_wht.fits'

csv444 = glob.glob(str(data_dir)+'/uds*444*_wcs.csv')[0]
csv770 = glob.glob(str(data_dir)+'/uds*770*_wcs.csv')[0]

# initialize drizzler; also reads the associated _wcs.csv files (rate files wcs/header information)
dpsf_444 = DrizzlePSF(driz_image=str(fsci_444), csv_file=csv444)
dpsf_770 = DrizzlePSF(driz_image=str(fsci_770), csv_file=csv770)

# map unique detector overlaps; keep only footprints overlapping the target mosaic
prm_444 = PSFRegionMap.from_footprints(dpsf_444.footprint, name='F444W').overlay_with(dpsf_444.driz_footprint)
prm_770 = PSFRegionMap.from_footprints(dpsf_770.footprint, name='F770W').overlay_with(dpsf_444.driz_footprint)

prm_kern = prm_444.overlay_with(prm_770)

psf_dir = Path('../data/PSF')
stpsf_444 = 'UDS_NRC.._F444W_OS4_GRID25'
stpsf_770 = 'UDS_MIRI_F770W_OS4_GRID9'
size = 2.0

# centroid positions of the regions: DONT drop poins here bc PSF list will be off
pos = [np.squeeze(p.xy) for p in prm_kern.regions.geometry.centroid]

# load webb psfs
dpsf_444.epsf_obj.load_jwst_stdpsf(local_dir=str(psf_dir), filter_pattern=stpsf_444)
dpsf_770.epsf_obj.load_jwst_stdpsf(local_dir=str(psf_dir), filter_pattern=stpsf_770)

# drizzle at centroid positions, size of stamp in arcsec
prm_444.psfs = dpsf_444.get_psf_radec(pos, size=size)
prm_770.psfs = dpsf_770.get_psf_radec(pos, size=size)

# store the PSFs + region maps
prm_444.to_file(data_dir / f'{field}-f444w_psf.geojson')
prm_770.to_file(data_dir / f'{field_miri}-f770w_psf.geojson')

# match kernels
# @@@ need a better way to determine best fft window shape
# @@@ clunky way to deal with different pixel scales, include zooming in matching kernel
# @@@ No PSF found, position possibly outside footprint for 34.38666666666666, -5.243333333333333 in filter UDS_MIRI_F770W_OS4_GRID9. Returning empty output.
# in this case, return nearest PSF, not empty 
from scipy.ndimage import zoom
from scipy.ndimage import gaussian_filter
scale = round(dpsf_770.driz_pscale/dpsf_444.driz_pscale)
nsize = prm_444.psfs.shape[-1]
odd = nsize & 1
kernels = [utils.matching_kernel(psf_444,
                    zoom(psf_770, scale, order=1, prefilter=False)[odd:odd+nsize,odd:odd+nsize],
           recenter=True) 
           for psf_444, psf_770 in zip(prm_444.psfs, prm_770.psfs)
]

prm_kern.psfs = np.asarray(kernels)
prm_kern.to_file(data_dir / f'{field_miri}-f444w_kernel_f770w.geojson')

In [ ]:
from pathlib import Path
from astropy.io import fits
from mophongo.psf_map import PSFRegionMap
from mophongo.fit import FitConfig
from mophongo.catalog import Catalog
from mophongo import pipeline
from matplotlib import pyplot as plt

data_dir = Path('../data')
field = 'uds-test2'
#field = 'uds-ahstar'
#field = 'uds-ahtwin'
#field = 'uds-bright'
field = 'uds-medium'
#field = 'uds-large'
field = 'uds-half'

field_miri = field+'-80mas'
#field_miri = field 

prm_444 = PSFRegionMap.from_geojson(str(data_dir / f'{field}-f444w_psf.geojson'))
prm_770 = PSFRegionMap.from_geojson(str(data_dir / f'{field_miri}-f770w_psf.geojson'))
prm_kern = PSFRegionMap.from_geojson(str(data_dir / f'{field_miri}-f444w_kernel_f770w.geojson'))

fsci_444 = data_dir / f'{field}-f444w_sci.fits'
fwht_444 = data_dir / f'{field}-f444w_wht.fits'

# read in LW selected catalog
fseg_LW = data_dir / f'{field}-LW_seg.fits'
#cat444 = Catalog.from_fits(sci_444, wht_444, segmap=seg_LW, estimate_ivar=True)

# MIRI images
field_miri = field+'-80mas'
fsci_770 = data_dir / f'{field_miri}-f770w_sci.fits'
fwht_770 = data_dir / f'{field_miri}-f770w_wht.fits'

print(fsci_770, fwht_770)
#tmpl_444 = fits.getdata(fsci_444)
# sci_770 = fits.getdata(fsci_770)
# wht_770 = fits.getdata(fwht_770)
segmap = fits.getdata(fseg_LW)

params = {'background_filter_sigma':32.0}
cat770 = Catalog.from_fits(fsci_770, fwht_770, params=params, estimate_background=True, estimate_ivar=True)
fig, ax = cat770.plot_bg(nbin=1, fac=0.5)
# plt.show()
params = {'background_filter_sigma':48.0}
cat444 = Catalog.from_fits(fsci_444, fwht_444, segmap=segmap, params=params)
# fig, ax = cat444.plot_bg(nbin=1, fac=0.01)
# plt.show()

In [ ]:
# first fit, no shifts: first image is template, 2nd and on the fitting images
config = FitConfig(fit_astrometry_niter=2,
                   scene_minimum_bright=5,
                   aperture_diam=0.5)

pipe = pipeline.Pipeline([cat444.sci, cat770.sci], segmap,
                                weights=[None, cat770.ivar],
                                catalog=cat444.table,
                                psfs=[None, prm_770],
                                kernels=[None, prm_kern],
                                wcs=[cat444.wcs, cat770.wcs])
table, res = pipe.run(config=config)


In [ ]:
tmpls = pipe.all_templates[0]
scenes = pipe.all_scenes[0]
for s in scenes:
    print(s.id, len(s.templates),s.is_bright.sum(), *s.shift_at(*s.shift_basis[1]))
#print(pipe.config.aperture_diam)
#print(tmpls[0].slices_cutout)
#tmpls[0].input_position_cutout
#tmpls[0].data


In [ ]:
#scenes[99].plot(cat444.sci, segmap, display_sig=1)

#scenes[65].plot(cat444.sci, segmap, display_sig=8)
#plt.imshow(scenes[0].model_image(),vmin=-1,vmax=1,cmap='gray')
#len(scenes[1].templates)
x, y = scenes[0].shift_basis[1]
wcs_444[0].wcs_pix2world([[x, y]], 0)

In [ ]:
from astropy.nddata import block_reduce
import numpy as np

res = block_reduce(pipe.residuals[0], (2, 2), np.sum)
mod = cat770.sci - res
plt.imshow( cat770.sci-mod, origin='lower',vmin=-1, vmax=1, cmap='gray')


In [ ]:
#fig, ax = pipe.plot_result(scene_id=15)
#fig, ax = pipe.plot_result(display_sig=2)
#fig, ax = pipe.plot_result(source_id=150067)
#fits.writeto('res.fits',res0[0])
#res0
img = tmpls[0].data
_ = plt.imshow(img, origin='lower', cmap='gray', vmin=-0.001, vmax=0.001)
plt.show()
_ = plt.imshow(np.gradient(img)[0], origin='lower', cmap='gray', vmin=-0.001, vmax=0.001)
plt.show()
_ = plt.imshow(np.gradient(img)[1], origin='lower', cmap='gray', vmin=-0.001, vmax=0.001)
plt.show()



In [ ]:
_ = plt.hist(table['flux_1']/table['err_1'],range=[-5,5], bins=50, log=True)

In [ ]:
table
#table['flux_1'].pprint(max_lines=-1)

In [ ]:
from pathlib import Path
from mophongo.catalog import find_saturated_stars

cat_dir = Path('/Users/ivo/Astro/PROJECTS/MINERVA/data/n2.2_m2.0_v1.0')
cat_file = 'MINERVA-UDS_n2.2_m2.0_v1.0_LW_Kf444w_SUPER_CATALOG.fits'

img_dir = Path('/Users/ivo/Astro/PROJECTS/MINERVA/data/v2')
img_file='uds-sbkgsub-v2.0-80mas-f770w_drz_wht.fits'


fits.getheader(img_dir / img_file)  # get header of the first extension

import os, requests
from astropy.io import fits

fn  = "jw01837001001_06101_00002_mirimage_rate.fits"
url = f"https://mast.stsci.edu/api/v0.1/Download/file?uri=mast:JWST/product/{fn}"

with fits.open(url, use_fsspec=True) as hdul:  # add fsspec_kwargs={"headers": hdrs} if needed
#    print(hdul[0].header["INSTRUME"], hdul[0].header.get("DATE-OBS"))
    hdr0 = hdul[0].header
    hdr1 = hdul[1].header


In [ ]:
from pathlib import Path
from mophongo.utils import write_wcs_csv

img_dir = Path('/Users/ivo/Astro/PROJECTS/MINERVA/data/v2')
img_file = 'uds-sbkgsub-v2.0-80mas-f770w_drz_wht.fits'

#write_wcs_csv_from_mosaic(img_dir / img_file)
write_wcs_csv(img_dir / 'uds-sbkgsub-v2.0-80mas-f770w_drz_sci.fits')